# User Health Knowledge for Welly AI

## Review / Objective
This notebook transforms patient-level nutrition and health data into structured knowledge for chatbot and RAG use. The workflow reviews the source dataset, prepares derived health signals, constructs a compact knowledge table, and exports CSV and JSON outputs to `../data/knowledge/`.


## Data loading
Load the patient dataset from `../data/Personalized_Diet_Recommendations.csv`, confirm that the file exists, and preview the raw records.


In [1]:
from pathlib import Path
import json
import re

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

DATA_PATH = Path("../data/Personalized_Diet_Recommendations.csv")
OUTPUT_DIR = Path("../data/knowledge")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Source file not found: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)

print(f"Loaded dataset from: {DATA_PATH}")
print(f"Dataset shape: {df.shape}")
display(df.head())


Loaded dataset from: ../data/Personalized_Diet_Recommendations.csv
Dataset shape: (5000, 30)


,Patient_ID,Age,Gender,Height_cm,Weight_kg,BMI,Chronic_Disease,Blood_Pressure_Systolic,Blood_Pressure_Diastolic,Cholesterol_Level,Blood_Sugar_Level,Genetic_Risk_Factor,Allergies,Daily_Steps,Exercise_Frequency,Sleep_Hours,Alcohol_Consumption,Smoking_Habit,Dietary_Habits,Caloric_Intake,Protein_Intake,Carbohydrate_Intake,Fat_Intake,Preferred_Cuisine,Food_Aversions,Recommended_Calories,Recommended_Protein,Recommended_Carbs,Recommended_Fats,Recommended_Meal_Plan
0,P00001,56,Other,163,66,24.84,NaN,175,75,219,124,No,NaN,11452,5,7.6,No,Yes,Vegetarian,2593,105,179,143,Western,NaN,2150,108,139,145,High-Protein Diet
1,P00002,69,Female,171,114,38.99,NaN,155,72,208,72,No,NaN,12962,1,6.3,Yes,No,Vegetarian,1852,69,315,75,Mediterranean,NaN,1527,74,266,80,Balanced Diet
2,P00003,46,Female,172,119,40.22,NaN,137,101,171,145,No,Gluten Intolerance,7898,3,9.9,No,No,Vegetarian,2737,183,103,148,Western,Sweet,2359,180,145,143,High-Protein Diet
3,P00004,32,Female,197,118,30.41,NaN,148,91,258,235,No,Nut Allergy,6602,5,4.2,No,No,Vegetarian,3289,135,371,120,Western,NaN,2858,137,378,135,High-Protein Diet
4,P00005,60,Female,156,109,44.79,Hypertension,160,109,260,248,Yes,NaN,9810,2,9.0,No,No,Regular,2405,167,298,48,Indian,Spicy,1937,166,317,56,High-Protein Diet


In [2]:
shape_summary = pd.DataFrame(
    {
        "metric": ["rows", "columns"],
        "value": [df.shape[0], df.shape[1]],
    }
)

missing_summary = (
    df.isna()
    .sum()
    .rename("missing_values")
    .to_frame()
    .assign(missing_pct=lambda x: (x["missing_values"] / len(df) * 100).round(2))
    .sort_values(["missing_values", "missing_pct"], ascending=False)
)

print("Columns:")
for column in df.columns:
    print(f"- {column}")

print("\nShape summary:")
display(shape_summary)

print("Missing values summary:")
display(missing_summary)


Columns:
- Patient_ID
- Age
- Gender
- Height_cm
- Weight_kg
- BMI
- Chronic_Disease
- Blood_Pressure_Systolic
- Blood_Pressure_Diastolic
- Cholesterol_Level
- Blood_Sugar_Level
- Genetic_Risk_Factor
- Allergies
- Daily_Steps
- Exercise_Frequency
- Sleep_Hours
- Alcohol_Consumption
- Smoking_Habit
- Dietary_Habits
- Caloric_Intake
- Protein_Intake
- Carbohydrate_Intake
- Fat_Intake
- Preferred_Cuisine
- Food_Aversions
- Recommended_Calories
- Recommended_Protein
- Recommended_Carbs
- Recommended_Fats
- Recommended_Meal_Plan

Shape summary:


,metric,value
0,rows,5000
1,columns,30


Missing values summary:


,missing_values,missing_pct
Allergies,3497,69.94
Chronic_Disease,2043,40.86
Food_Aversions,1225,24.50
Patient_ID,0,0.00
Age,0,0.00
Gender,0,0.00
Height_cm,0,0.00
Weight_kg,0,0.00
BMI,0,0.00
Blood_Pressure_Systolic,0,0.00


## Data preparation
Standardize the source schema, derive `BMI_Category` using Asian BMI thresholds, and build a rule-based `Health_Profile_Summary` from BMI, blood pressure, blood sugar, cholesterol, and daily steps. The summary is intended for knowledge tagging and retrieval support, not clinical diagnosis.


In [3]:
REQUIRED_SOURCE_COLUMNS = [
    "Patient_ID",
    "Age",
    "Gender",
    "Height_cm",
    "Weight_kg",
    "BMI",
    "Blood_Pressure_Systolic",
    "Blood_Pressure_Diastolic",
    "Blood_Sugar_Level",
    "Cholesterol_Level",
    "Daily_Steps",
    "Recommended_Calories",
    "Recommended_Protein",
    "Recommended_Carbs",
    "Recommended_Fats",
    "Recommended_Meal_Plan",
]

COLUMN_ALIASES = {
    "Patient_ID": ["patient_id", "patientid", "user_id", "member_id"],
    "Age": ["age", "patient_age"],
    "Gender": ["gender", "sex"],
    "Height_cm": ["height_cm", "heightcm", "height"],
    "Weight_kg": ["weight_kg", "weightkg", "weight"],
    "BMI": ["bmi", "body_mass_index"],
    "Blood_Pressure_Systolic": ["blood_pressure_systolic", "systolic_bp", "sbp"],
    "Blood_Pressure_Diastolic": ["blood_pressure_diastolic", "diastolic_bp", "dbp"],
    "Blood_Sugar_Level": ["blood_sugar_level", "blood_glucose", "glucose_level"],
    "Cholesterol_Level": ["cholesterol_level", "total_cholesterol", "cholesterol"],
    "Daily_Steps": ["daily_steps", "steps", "step_count"],
    "Recommended_Calories": ["recommended_calories", "target_calories"],
    "Recommended_Protein": ["recommended_protein", "target_protein"],
    "Recommended_Carbs": ["recommended_carbs", "target_carbs", "recommended_carbohydrates"],
    "Recommended_Fats": ["recommended_fats", "target_fats"],
    "Recommended_Meal_Plan": ["recommended_meal_plan", "meal_plan", "diet_plan"],
}

def normalize_column_name(name):
    return re.sub(r"[^a-z0-9]+", "", str(name).strip().lower())

def resolve_columns(frame, canonical_columns, aliases=None):
    aliases = aliases or {}
    normalized_lookup = {normalize_column_name(column): column for column in frame.columns}
    resolved = {}
    missing = []

    for canonical in canonical_columns:
        candidates = [canonical, *aliases.get(canonical, [])]
        match = None
        for candidate in candidates:
            match = normalized_lookup.get(normalize_column_name(candidate))
            if match:
                break
        if match is None:
            missing.append(canonical)
        else:
            resolved[canonical] = match

    if missing:
        raise KeyError(f"Missing required columns after automatic matching: {missing}")

    return resolved

def classify_bmi_asian(bmi):
    if pd.isna(bmi):
        return "Unknown"
    if bmi < 18.5:
        return "Underweight"
    if bmi < 23.0:
        return "Normal"
    if bmi < 25.0:
        return "Overweight"
    if bmi < 30.0:
        return "Obese Level 1"
    return "Obese Level 2"

def classify_blood_pressure(systolic, diastolic):
    if pd.isna(systolic) or pd.isna(diastolic):
        return "Unknown"
    if systolic >= 140 or diastolic >= 90:
        return "Stage 2 hypertension range"
    if systolic >= 130 or diastolic >= 80:
        return "Stage 1 hypertension range"
    if systolic >= 120 and diastolic < 80:
        return "Elevated"
    return "Normal"

def classify_blood_sugar(value):
    if pd.isna(value):
        return "Unknown"
    if value >= 126:
        return "High"
    if value >= 100:
        return "Borderline high"
    return "Normal"

def classify_cholesterol(value):
    if pd.isna(value):
        return "Unknown"
    if value >= 240:
        return "High"
    if value >= 200:
        return "Borderline high"
    return "Desirable"

def classify_activity(steps):
    if pd.isna(steps):
        return "Unknown"
    if steps < 5000:
        return "Low activity"
    if steps < 7500:
        return "Lightly active"
    if steps < 10000:
        return "Moderately active"
    return "Active"

def calculate_risk_score(bmi_category, bp_status, sugar_status, cholesterol_status, activity_status):
    score = 0
    score += {"Underweight": 1, "Normal": 0, "Overweight": 1, "Obese Level 1": 2, "Obese Level 2": 3}.get(bmi_category, 0)
    score += {"Normal": 0, "Elevated": 1, "Stage 1 hypertension range": 2, "Stage 2 hypertension range": 3}.get(bp_status, 0)
    score += {"Normal": 0, "Borderline high": 1, "High": 3}.get(sugar_status, 0)
    score += {"Desirable": 0, "Borderline high": 1, "High": 2}.get(cholesterol_status, 0)
    score += {"Active": 0, "Moderately active": 0, "Lightly active": 1, "Low activity": 2}.get(activity_status, 0)
    return score

def summarize_overall_risk(score):
    if score >= 8:
        return "high"
    if score >= 4:
        return "moderate"
    return "lower"

def build_health_profile_summary(row, resolved_columns):
    bmi = row[resolved_columns["BMI"]]
    systolic = row[resolved_columns["Blood_Pressure_Systolic"]]
    diastolic = row[resolved_columns["Blood_Pressure_Diastolic"]]
    blood_sugar = row[resolved_columns["Blood_Sugar_Level"]]
    cholesterol = row[resolved_columns["Cholesterol_Level"]]
    daily_steps = row[resolved_columns["Daily_Steps"]]

    bmi_category = classify_bmi_asian(bmi)
    bp_status = classify_blood_pressure(systolic, diastolic)
    sugar_status = classify_blood_sugar(blood_sugar)
    cholesterol_status = classify_cholesterol(cholesterol)
    activity_status = classify_activity(daily_steps)

    risk_score = calculate_risk_score(
        bmi_category=bmi_category,
        bp_status=bp_status,
        sugar_status=sugar_status,
        cholesterol_status=cholesterol_status,
        activity_status=activity_status,
    )
    overall_risk = summarize_overall_risk(risk_score)

    return (
        f"Overall {overall_risk} cardiometabolic risk. "
        f"BMI: {bmi_category} ({bmi:.2f}). "
        f"Blood pressure: {bp_status} ({int(round(systolic))}/{int(round(diastolic))} mmHg). "
        f"Blood sugar: {sugar_status} ({int(round(blood_sugar))} mg/dL). "
        f"Cholesterol: {cholesterol_status} ({int(round(cholesterol))} mg/dL). "
        f"Daily activity: {activity_status} ({int(round(daily_steps)):,} steps/day)."
    )

resolved_columns = resolve_columns(df, REQUIRED_SOURCE_COLUMNS, COLUMN_ALIASES)

prepared_df = df.copy()
prepared_df["BMI_Category"] = prepared_df[resolved_columns["BMI"]].apply(classify_bmi_asian)
prepared_df["Health_Profile_Summary"] = prepared_df.apply(
    build_health_profile_summary,
    axis=1,
    resolved_columns=resolved_columns,
)

print("Resolved source columns:")
display(pd.Series(resolved_columns, name="matched_column").rename_axis("canonical_column").to_frame())

print("BMI category distribution:")
display(prepared_df["BMI_Category"].value_counts().rename_axis("BMI_Category").to_frame("patients"))

display(
    prepared_df[
        [
            resolved_columns["Patient_ID"],
            resolved_columns["BMI"],
            "BMI_Category",
            "Health_Profile_Summary",
        ]
    ].head()
)


Resolved source columns:


,matched_column
canonical_column,
Patient_ID,Patient_ID
Age,Age
Gender,Gender
Height_cm,Height_cm
Weight_kg,Weight_kg
BMI,BMI
Blood_Pressure_Systolic,Blood_Pressure_Systolic
Blood_Pressure_Diastolic,Blood_Pressure_Diastolic
Blood_Sugar_Level,Blood_Sugar_Level


BMI category distribution:


,patients
BMI_Category,
Obese Level 2,2009
Obese Level 1,1070
Normal,897
Underweight,591
Overweight,433


,Patient_ID,BMI,BMI_Category,Health_Profile_Summary
0,P00001,24.84,Overweight,Overall moderate cardiometabolic risk. BMI: Ov...
1,P00002,38.99,Obese Level 2,Overall moderate cardiometabolic risk. BMI: Ob...
2,P00003,40.22,Obese Level 2,Overall high cardiometabolic risk. BMI: Obese ...
3,P00004,30.41,Obese Level 2,Overall high cardiometabolic risk. BMI: Obese ...
4,P00005,44.79,Obese Level 2,Overall high cardiometabolic risk. BMI: Obese ...


## Knowledge construction
Keep the most relevant fields for downstream retrieval, preserve a consistent canonical schema, and preview the final knowledge table.


In [4]:
KNOWLEDGE_COLUMNS = [
    "Patient_ID",
    "Age",
    "Gender",
    "Height_cm",
    "Weight_kg",
    "BMI",
    "BMI_Category",
    "Blood_Pressure_Systolic",
    "Blood_Pressure_Diastolic",
    "Blood_Sugar_Level",
    "Cholesterol_Level",
    "Health_Profile_Summary",
    "Recommended_Calories",
    "Recommended_Protein",
    "Recommended_Carbs",
    "Recommended_Fats",
    "Recommended_Meal_Plan",
]

selected_columns = [resolved_columns.get(column, column) for column in KNOWLEDGE_COLUMNS]
knowledge_df = prepared_df[selected_columns].copy()
knowledge_df = knowledge_df.rename(
    columns={resolved_columns[canonical]: canonical for canonical in resolved_columns if canonical in KNOWLEDGE_COLUMNS}
)
knowledge_df = knowledge_df[KNOWLEDGE_COLUMNS]

print(f"Knowledge table shape: {knowledge_df.shape}")
display(knowledge_df.head())


Knowledge table shape: (5000, 17)


,Patient_ID,Age,Gender,Height_cm,Weight_kg,BMI,BMI_Category,Blood_Pressure_Systolic,Blood_Pressure_Diastolic,Blood_Sugar_Level,Cholesterol_Level,Health_Profile_Summary,Recommended_Calories,Recommended_Protein,Recommended_Carbs,Recommended_Fats,Recommended_Meal_Plan
0,P00001,56,Other,163,66,24.84,Overweight,175,75,124,219,Overall moderate cardiometabolic risk. BMI: Ov...,2150,108,139,145,High-Protein Diet
1,P00002,69,Female,171,114,38.99,Obese Level 2,155,72,72,208,Overall moderate cardiometabolic risk. BMI: Ob...,1527,74,266,80,Balanced Diet
2,P00003,46,Female,172,119,40.22,Obese Level 2,137,101,145,171,Overall high cardiometabolic risk. BMI: Obese ...,2359,180,145,143,High-Protein Diet
3,P00004,32,Female,197,118,30.41,Obese Level 2,148,91,235,258,Overall high cardiometabolic risk. BMI: Obese ...,2858,137,378,135,High-Protein Diet
4,P00005,60,Female,156,109,44.79,Obese Level 2,160,109,248,260,Overall high cardiometabolic risk. BMI: Obese ...,1937,166,317,56,High-Protein Diet


## Output saving
Export the knowledge table to both CSV and JSON so it can be reused in ETL, retrieval, and chatbot workflows.


In [5]:
csv_output_path = OUTPUT_DIR / "user_health_knowledge.csv"
json_output_path = OUTPUT_DIR / "user_health_knowledge.json"

knowledge_df.to_csv(csv_output_path, index=False)

with json_output_path.open("w", encoding="utf-8") as output_file:
    json.dump(knowledge_df.to_dict(orient="records"), output_file, ensure_ascii=False, indent=2)

print(f"Saved CSV to: {csv_output_path}")
print(f"Saved JSON to: {json_output_path}")
print(f"Knowledge records exported: {len(knowledge_df):,}")


Saved CSV to: ../data/knowledge/user_health_knowledge.csv
Saved JSON to: ../data/knowledge/user_health_knowledge.json
Knowledge records exported: 5,000
